# Solving RLWE with Grover's algorithm

In this tutorial, we demonstrate how to solve the Ring Learning With Errors (RLWE) problem using Grover's algorithm. The RLWE problem forms the mathematical foundation of many modern post-quantum cryptographic schemes, including the NIST standard **ML-KEM**.

## The RLWE Problem

The problem is defined over the polynomial ring $\mathcal{R}_q = \mathbb{Z}_q[X] / (X^n + 1)$, where $n$ is a power of 2 and $q$ is a prime modulus. 

In an RLWE instance, we are given a uniformly random public polynomial $a \in \mathcal{R}_q$ and a polynomial $t \in \mathcal{R}_q$ computed as:
$$ t = a \cdot s + e $$
where $s, e \in \mathcal{R}_q$ are the secret and error polynomials, respectively. Crucially, the coefficients of $s$ and $e$ are sampled from a distribution that only produces "small" integers. The adversary's goal is to recover the secret $s$ given the public pair $(a, t)$.

Our setup essentially mirrors ML-KEM with $k=1$, but generalizes it to arbitrary power-of-two dimensions $n$. While ML-KEM fixes $n=256$, allowing variable $n$ enables us to construct small, toy-sized cryptographic instances where the complete Grover quantum computation can still be evaluated and simulated effectively on a classical machine.

## The Number Theoretic Transform (NTT)

Polynomial multiplication in $\mathcal{R}_q$ requires $\mathcal{O}(n^2)$ coefficient multiplications if computed naively. To perform this efficiently, we use the Number Theoretic Transform (NTT), the finite-field equivalent of the Fast Fourier Transform (FFT).

If a *complete* NTT exists, the ring $\mathcal{R}_q$ splits into $n$ separate scalar components $\prod \mathbb{Z}_q$, and polynomial multiplication becomes completely element-wise. However, a complete NTT requires $q \equiv 1 \pmod{2n}$ to ensure the existence of a $2n$-th primitive root of unity modulo $q$.
    
In cryptographic schemes like ML-KEM, we intentionally choose $q=3329$ and $n=256$. Since $2n = 512$ does not divide $q-1 = 3328$, a complete NTT does not exist! However, $n=256$ *does* divide $q-1$, which guarantees the existence of an $n$-th primitive root of unity. This allows us to compute an **incomplete NTT**. 
    
The incomplete NTT maps our polynomial into $n/2$ components of degree 1 (instead of degree 0 scalars):
$$ \mathcal{R}_q \cong \prod_{i=0}^{n/2-1} \mathbb{Z}_q[X] / (X^2 - \zeta_{2i+1}) $$
In this domain, the polynomial multiplication $\hat{t} = \hat{a}\circ \hat{s} + \hat{e}$ is computed pairwise on degree-1 polynomials (so-called base-case multiplications), avoiding cross-terms from different components and reducing the multiplication complexity to $\mathcal{O}(n \log n)$.

### Classical implementation of NTT

In [1]:
import numpy as np


def bitrevm(r, m):
    """Compute the bit-reversal of r with respect to m bits."""
    rev = 0
    for _ in range(m):
        rev = (rev << 1) | (r & 1)
        r >>= 1
    return rev


def ntt(f, q, root):
    """
    Computes the NTT representation f_hat of the given polynomial f.
    f: 1-D numpy array of polynomial coefficients.
    q: The modulus.
    root: The n-th primitive root of unity modulo q.
    """
    n = len(f)
    m = n.bit_length() - 1
    
    f_hat = f.copy() % q 
    i = 1
    
    length = n // 2
    while length >= 2:
        for start in range(0, n, 2 * length):
            zeta = pow(int(root), int(bitrevm(i, m - 1)), int(q))
            i += 1
            
            for j in range(start, start + length):
                t = (zeta * f_hat[j + length]) % q
                f_hat[j + length] = (f_hat[j] - t) % q
                f_hat[j] = (f_hat[j] + t) % q
                
        length //= 2
        
    return f_hat

Let's test this classical function to ensure everything works correctly.

In [2]:
n, q, root = 4, 13, 5
a = np.array([3, 1, 4, 9])
ntt(a, q, root)

array([10,  7,  9,  8])

### Quantum implementation of NTT

We translate the classical NTT directly into a quantum representation using Qrisp's `QuantumArray` equipped with `QuantumModulus` data types. These abstract arrays seamlessly represent vectors in $\mathcal{R}_q \cong \mathbb{Z}_q^n$.

The quantum implementation relies on Qrisp's JAX-traceable tools and loops to iteratively evaluate the transform in-place across the quantum array.

In [3]:
from qrisp import h, conjugate, control, invert, measure, QuantumArray, QuantumBool, QuantumModulus, z

n, q = 4, 13
v = QuantumArray(QuantumModulus(q), shape=(n,))
v[:] = np.array([3, 1, 4, 9])
print(v)

{OutcomeArray([3, 1, 4, 9]): 1.0}                                               


We can now implement the quantum NTT function `qntt`, which acts in-place directly on our `QuantumArray`.

To ensure compatibility with Qrisp's JAX-based compilation pipeline, the classical operations must be written using JAX-traceable logic. Rather than standard Python `for` and `while` loops, we use `jax.lax.fori_loop` and `jax.lax.while_loop`. This allows Qrisp's tracer to evaluate and unroll them seamlessly during quantum compilation. 

Let's explicitly define the JAX-compatible versions of `bitrevm` and our modular exponentiation helper `modpow` here to see how classical loops are translated:

In [ ]:
import operator

import jax
from jax import lax
import jax.numpy as jnp

from qrisp.jasp import q_while_loop, jaspify, jrange, terminal_sampling

@jax.jit
def bitrevm(r, m):
    """Compute the bit-reversal of r with respect to m bits (JAX version)."""
    x = jnp.asarray(r)

    def body(i, val):
        rev, x = val
        rev = (rev << 1) | (x & 1)
        x = x >> 1
        return (rev, x)

    rev, _ = lax.fori_loop(0, m, body, (0, x))
    return rev


@jax.jit
def modpow(a, x, q):
    """Computes (a ** x) % q (JAX version)"""
    a = jnp.asarray(a) % jnp.asarray(q)
    exp = jnp.asarray(x)
    mod = jnp.asarray(q)

    init = (a, exp, jnp.ones_like(a))

    def cond_fn(state):
        _, e, _ = state
        return jnp.any(e > 0)

    def body_fn(state):
        base, e, res = state
        res = jnp.where((e & 1) != 0, (res * base) % mod, res)
        base = (base * base) % mod
        e = e >> 1
        return (base, e, res)

    _, _, result = lax.while_loop(cond_fn, body_fn, init)
    return result


# NIST FIPS 203, Algorithm 9, quantum version
# Efficient O(n log n) implementation
def qntt(f: QuantumArray, root: int) -> None:
    """Computes the number theoretic transform (NTT) in-place."""

    n = f.shape[0]
    m = n.bit_length() - 1
    q = f.qtype.modulus

    def cond_fun_inner(val):
        start, len_, i, f = val
        return start < n

    def body_fun_inner(val):
        start, len_, i, f = val

        # Replace bitrev7 by general procedure for arbitrary n
        zeta = modpow(root, bitrevm(i, m - 1), q)

        for j in jrange(start, start + len_):
            # Reversible implementation of steps 8-10
            f[j + len_] *= zeta
            f[j] += f[j + len_]
            f[j + len_] *= q - 2  # the same as *= -2
            f[j + len_] += f[j]

        return start + 2 * len_, len_, i + 1, f

    def cond_fun_outer(val):
        len_, i, f = val
        return len_ >= 2

    def body_fun_outer(val):
        len_, i, f = val

        _, _, i, _ = q_while_loop(cond_fun_inner, body_fun_inner, (0, len_, i, f))

        return len_ // 2, i, f

    q_while_loop(cond_fun_outer, body_fun_outer, (n // 2, 1, f))

Let's test our quantum NTT structure to verify it behaves exactly like its classical counterpart. 

We will compute the NTT of the same vector $a=(3,1,4,9)\in\mathbb Z_{13}^4$ with respect to the $4$-th root of unity $\zeta=5$ modulo $13$. To do this, we encode the classical array into a `QuantumArray` of `QuantumModulus` integers, apply `qntt` in-place within a `@jaspify`-decorated function, and `measure` the results.

In [5]:
@jaspify
def main():

    n, q, root = 4, 13, 5
    a = jnp.array([3, 1, 4, 9])

    qa = QuantumArray(QuantumModulus(q), shape=(n,))
    qa[:] = a

    qntt(qa, root)

    return measure(qa)

print(main())

[10.  7.  9.  8.]                                                               


The result exactly matches the underlying classical mathematical evaluation!

Now, let's test a multiplication algorithm leveraging this NTT structure using the `multiply_qntts` primitive, which executes the polynomial base-case multiplication operations pointwise across the transformed components.

In [6]:
from qrisp.alg_primitives.ntt import multiply_qntts

@jaspify
def main():

    n, q, root = 4, 13, 5
    a = jnp.array([3, 1, 4, 9])
    b = np.array([1, 2, 3, 4])

    qa = QuantumArray(QuantumModulus(q), shape=(n,))
    qa[:] = a

    res = multiply_qntts(qa, b, root)

    return measure(res)

print(main())

[0. 7. 1. 4.]                                                                   


We now possess all mathematical primitives and quantum transformations necessary to construct the Grover attack!

## The Grover attack on RLWE

### The Grover oracle

A Grover oracle must uniquely mark the solution state. Since $t = a \cdot s + e$, we know that for the correct secret $s$, the difference $r = a \cdot s - t$ equals the error $e$. 

By problem definition, $e$ is strictly "small". Therefore, our oracle computes $r$ and checks if its coefficients are bounded by a threshold $b_e$. Specifically, it checks if the value $v$ satisfies $v \le b_e$ or $v \ge q - b_e$ (accounting for negative values modulo $q$). The state receives a phase of $-1$ if all coefficients perfectly satisfy this condition.

To achieve this in Qrisp, we evaluate boolean expressions out-of-place and use Qrisp's **Quantum Function Injection operator (`<<`)** to turn them into in-place assignments. This mechanism dynamically targets workspace variables, making it exceptionally elegant to recursively uncompute intermediate values using `conjugate`.

In [7]:
def tag_if_small(v: QuantumArray, b_e) -> None:
    """Check if the elements of v are within the bounds defined by b_e."""
    n = v.shape[0]
    flag_l = QuantumArray(QuantumBool(), shape=(n,)) 
    flag_r = QuantumArray(QuantumBool(), shape=(n,))
    flag_lr = QuantumArray(QuantumBool(), shape=(n,))
    flag_all = QuantumBool()

    # We use Qrisp's quantum function injection operator "<<" to turn 
    # out-of-place evaluations into in-place reversible targets for conjugation.
    small_inj_l = flag_l << (lambda v: v <= b_e)
    small_inj_r = flag_r << (lambda v: v >= q - b_e)
    or_inj = flag_lr << (lambda v, w: v | w)
    all_inj = flag_all << (lambda v: v.all())

    with conjugate(small_inj_l)(v):
        with conjugate(small_inj_r)(v):
            with conjugate(or_inj)(flag_l, flag_r):
                with conjugate(all_inj)(flag_lr):
                    z(flag_all)

    flag_l.delete()
    flag_r.delete()
    flag_lr.delete()
    flag_all.delete()

Let's test this oracle marking scheme!

We prepare a `QuantumArray` $v$ of `QuantumModulus` with $q=13$ in a uniform quantum superposition state: 
$$ \ket{\psi} = \sum_{v\in S}\ket{v} $$
for all $S\subset\mathbb{Z}_q^n$.

We define an additional `QuantumBool` acting as a flag. The flag is brought into the superposition $\frac{1}{\sqrt{2}} (\ket{0}+\ket{1})$ via a Hadamard gate. The `tag_if_small` oracle is then applied to $v$, controlled by the flag being in state $\ket{1}$. 

Because `tag_if_small` ultimately applies a $Z$-gate (a phase of $-1$) to the target, the controlled application selectively flips the phase of the flag's $\ket{1}$ state if and only if $v$ evaluates to "small". Thus, the state gets entangled as:
$$ \ket{\psi'} = \sum_{v\in S'}\ket{v}\otimes \frac{\ket{0}-\ket{1}}{\sqrt{2}} + \sum_{v\in S''}\ket{v}\otimes \frac{\ket{0}+\ket{1}}{\sqrt{2}} $$
where $S',S''\subset S$ are the disjoint subsets of small and large entries, respectively. 

Finally, applying a Hadamard gate on the flag maps the phase states basis back to the computational basis ($\frac{\ket{0}-\ket{1}}{\sqrt{2}} \mapsto \ket{1}$ and $\frac{\ket{0}+\ket{1}}{\sqrt{2}} \mapsto \ket{0}$), yielding the final state:
$$ \ket{\psi''} = \sum_{v\in S'}\ket{v}\otimes\ket{1} + \sum_{v\in S''}\ket{v}\otimes\ket{0} $$
This successfully maps the phase kickback into an observable amplitude flip.

In [8]:
def prep(v):
    for i in jrange(v.shape[0]):
        h(v[i][0])
        h(v[i][1])


@jaspify
def test_tag_if_small():

    v = QuantumArray(QuantumModulus(13), shape=(4,))
    prep(v)

    flag = QuantumBool()
    with conjugate(h)(flag):
        with control(flag[0]):
            tag_if_small(v, 2)
            
    return measure(v), measure(flag)

test_tag_if_small()

(Array([0., 1., 1., 0.], dtype=float64), Array(True, dtype=bool))

Because polynomial operations over $\mathcal{R}_q$ are resource-intensive to implement in standard representation cleanly, the Grover oracle evaluates the arithmetic internally in the NTT domain. 

Acting on a quantum state representing a candidate secret $\ket{s}$, the exact sequence is:

1. **Transform**: $\ket{s} \mapsto \ket{\hat{s}}$ (Quantum NTT)
2. **Multiply**: $\ket{\hat{r}} = \hat{a} \circ \ket{\hat{s}}$ (Multiplication in the NTT domain)
3. **Subtract**: $\ket{\hat{r}} \leftarrow \ket{\hat{r}} - \hat{t}$
4. **Inverse Transform**: $\ket{\hat{r}} \mapsto \ket{r}$ (Quantum Inverse NTT)
5. **Tag**: `tag_if_small(r)` (Mark if the residual error $r$ is bounded modulo $q$)
6. **Uncompute**: Steps 1-4 are uncomputed to keep the workspace clean.

Qrisp's powerful `conjugate` and `invert` blocks elegantly handle the uncomputation of all intermediate arithmetic states automatically!

In [9]:
# Use Qrisp's qnnt implemnentation which has a custom inversion
from qrisp.alg_primitives.ntt import qntt 

def inv_qntt(v, root):
    with invert():
        qntt(v, root)


def create_lwe_oracle(a_hat, t_hat, q, root, b_e):

    n = a_hat.shape[0]

    def oracle(s_hat):
        with conjugate(qntt)(s_hat, root): # s -> s_hat
            r_hat = QuantumArray(QuantumModulus(q), shape=(n,))
            inj_multiply_qntts = r_hat << (lambda s_hat, a_hat, root: multiply_qntts(s_hat, a_hat, root))

            with conjugate(inj_multiply_qntts)(s_hat, a_hat, root):

                with conjugate(operator.isub)(r_hat, t_hat):

                    with conjugate(inv_qntt)(r_hat, root): # r_hat -> r

                        tag_if_small(r_hat, b_e)

            r_hat.delete()

    return oracle

### Example: n=4, q=13

Let's construct a small random RLWE instance locally to test against. We isolate the parameters $n=4, q=13$, setting a primitive root $\zeta=5$.

We generate random standard polynomials $a, s, e$ subject to small values and execute their mathematical product to simulate the public RLWE challenge vector $t$.

In [4]:
from qrisp.alg_primitives.ntt import multiply_ntts, ntt, ntt_inv

def lwe_instance(n, q, root):
    a = np.array(np.random.randint(0, q, size=n))
    s = np.array(np.random.randint(0, 2, size=n)) # small secret
    e = np.array(np.random.randint(0, 2, size=n)) # small error

    a_hat = np.array(ntt(a, q, root))
    s_hat = np.array(ntt(s, q, root))
    e_hat = np.array(ntt(e, q, root))


    t_hat = multiply_ntts(a_hat, s_hat, q, root)
    t_hat += e_hat
    t = np.array(ntt_inv(t_hat, q, root))

    return a, s, e, t

n = 4
q = 13
root = 5

# Set the seed here for reproducible randomness
np.random.seed(42)

a, s, e, t = lwe_instance(n, q, root)

print("Public polynomial 'a':", a)
print("Secret polynomial 's':", s)
print("Error polynomial 'e': ", e)
print("Public polynomial 't':", t)

Public polynomial 'a': [ 6  3 12 10]
Secret polynomial 's': [1 0 0 0]
Error polynomial 'e':  [1 0 0 0]
Public polynomial 't': [ 7  3 12 10]


Before committing to amplitude amplification, we can check that our implemented `oracle` consistently tags the exact correct secret representation using the phase-flip flag technique outlined previously.

In [11]:
# Uase JAX's jnp.asarray to convert numpy arrays to JAX arrays for compatibility with Qrisp
a_hat = jnp.asarray(ntt(a, q, root))
t_hat = jnp.asarray(ntt(t, q, root))
s_hat = jnp.asarray(ntt(s, q, root))

oracle = create_lwe_oracle(a_hat, t_hat, q, root=root, b_e=2)

@terminal_sampling
def main():
    qs = QuantumArray(QuantumModulus(q), shape=(n,))
    qs[:] = jnp.asarray(s)

    qb = QuantumBool()
    h(qb)
    with control(qb):
        oracle(qs)
    h(qb)
    return qb

main()

{True: 1.0}

True indicates that the secret $s$ is tagged.

## Amplitude amplification

To resolve the RLWE instance and uncover the missing secret vector $s$, we define an initial state generation sequence that injects a uniform superposition across the "small" combinatorial search space of $s$.

Using this prepared superposition `state_func` intertwined with our evaluated `oracle_func`, Qrisp iteratively cascades Grover's `amplitude_amplification` operator over the bounded system until the amplitude peaks significantly on our true secret.

In [13]:
from qrisp import amplitude_amplification

oracle_func = create_lwe_oracle(a_hat, t_hat, q, root=root, b_e=2)

def state_func(qa):
    for i in jrange(3):
        h(qa[i][0])

@jaspify
def main(i):
    qs = QuantumArray(QuantumModulus(q), shape=(n,))
    state_func(qs)

    amplitude_amplification(qs, state_func, oracle_func, iter=i)

    return measure(qs)

main(1)

Array([1., 0., 0., 0.], dtype=float64)

And just like that, we have uncovered the quantumly-evaluated secret vector derived strictly from the generated public keys $a$ and $t$, cracking the synthesized classical ring instance!